# CornerScout - 07 Agente OpenAI y herramientas de solo lectura

Un solo agente responde preguntas de seguimiento sobre una sesion fija
mediante exactamente tres herramientas de solo lectura. El ciclo real usa
OpenAI function calling, valida argumentos con Pydantic, ejecuta únicamente
mediante `invoke_tool` y valida la respuesta final como `AgentAnswer`.

No hay multiagente, base vectorial, SQL, escritura, navegacion ni acceso a
archivos desde el agente o sus herramientas. `run_mock_agent` se conserva
exclusivamente para pruebas. El fallback de la ruta real es determinista.

> **Aviso de edición:** las salidas guardadas se preservan como evidencia
> histórica de la ejecución anterior, que solo ejercitó el agente simulado.
> No son resultados finales de este código hasta reejecutar el notebook
> completo desde un runtime limpio.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("openai") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])

print("Dependencia OpenAI disponible")

Dependencia OpenAI disponible


In [ ]:
from openai import OpenAI

print("OpenAI SDK disponible")

OpenAI SDK disponible


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Corner_Scout"
)

DATA_DIR = PROJECT_ROOT / "data"

assert PROJECT_ROOT.exists()
assert DATA_DIR.exists()
assert (PROJECT_ROOT / "analytics" / "io.py").is_file(), (
    "Falta copiar analytics/io.py a Google Drive"
)

os.environ["CORNERSCOUT_DATA_DIR"] = str(DATA_DIR)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics.io import data_dir, digest, write_json

DATA = data_dir()

print("Proyecto:", PROJECT_ROOT)
print("Datos:", DATA)

Proyecto: /content/drive/MyDrive/Corner_Scout
Datos: /content/drive/MyDrive/Corner_Scout/data


In [ ]:
import json
import os
import platform
import re
import time
from datetime import date, datetime, timezone
from importlib.metadata import version
from typing import Any, Callable, Literal

import jsonschema
import pandas as pd
import pydantic
from IPython.display import display
from pydantic import BaseModel, ConfigDict, Field

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except ImportError:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

DATA = data_dir()
INPUT = DATA / "processed" / "06_report"
OUT = DATA / "processed" / "07_agent"
OUT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
STAGE_VERSION = "07-readonly-agent-v2-openai"
RUN_REAL_LLM = bool(OPENAI_API_KEY)
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-5.6-terra")
OPENAI_TIMEOUT_SECONDS = 30
MAX_WALL_SECONDS = 45
MAX_TOOL_CALLS = 4
MAX_REAL_TURNS = 2
MAX_OUTPUT_TOKENS = 1600
MAX_SESSION_TOKENS = 6000

print({
    "run_id": RUN_ID,
    "stage": STAGE_VERSION,
    "run_real_llm": RUN_REAL_LLM,
    "openai_secret_loaded": bool(OPENAI_API_KEY),
    "python": platform.python_version(),
    "pydantic": pydantic.__version__,
    "jsonschema": version("jsonschema"),
})

{'run_id': '20260923T043911_797879Z', 'stage': '07-readonly-agent-v2-openai', 'run_real_llm': True, 'openai_secret_loaded': True, 'python': '3.13.15', 'pydantic': '2.13.5', 'jsonschema': '4.26.0'}


## 1. Precondicion y sesion reutilizada de 06

In [ ]:
contract06 = json.loads((INPUT / "contract.json").read_text(encoding="utf-8"))
assert contract06["contract_version"] == "06-tactical-report-v1"
assert contract06["final_approval_rate"] == 1.0

def artifact(name):
    return next(item for item in contract06["artifacts"] if item["file"] == name)

for name in ["evidence_schema.json", "report_schema.json", "evidence_cases.json", "report_runs.json"]:
    assert digest(INPUT / name) == artifact(name)["sha256"]

evidence_schema = json.loads((INPUT / "evidence_schema.json").read_text(encoding="utf-8"))
report_schema = json.loads((INPUT / "report_schema.json").read_text(encoding="utf-8"))
evidence_cases = json.loads((INPUT / "evidence_cases.json").read_text(encoding="utf-8"))
report_runs = json.loads((INPUT / "report_runs.json").read_text(encoding="utf-8"))
normal_evidence = evidence_cases["normal"]
normal_run = next(item for item in report_runs if item["case_id"] == "normal")
jsonschema.validate(normal_evidence, evidence_schema)
jsonschema.validate(normal_run["report"], report_schema)

SESSION_RIVAL = normal_evidence["rival"]
SESSION_CUTOFF = date.fromisoformat(normal_evidence["fecha_corte"])
evidence_by_id = {
    item["evidence_id"]: item
    for item in [*normal_evidence["indicadores"], *normal_evidence["limitaciones"],
                 *normal_evidence["resultados_modelo_promovidos"]]
}
SESSION_MEMORY = {
    "rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF,
    "history_match_ids": tuple(normal_evidence["history_match_ids"]),
    "indicators": tuple(normal_evidence["indicadores"]),
    "limitations": tuple(normal_evidence["limitaciones"]),
    "promoted_models": tuple(normal_evidence["resultados_modelo_promovidos"]),
    "evidence_by_id": evidence_by_id,
    "one_button_report": normal_run["report"],
}
assert SESSION_MEMORY["rival"] == "Barcelona" and len(SESSION_MEMORY["history_match_ids"]) == 8
print({"session_rival": SESSION_RIVAL, "session_cutoff": str(SESSION_CUTOFF),
       "evidence_items": len(evidence_by_id)})

{'session_rival': 'Barcelona', 'session_cutoff': '2016-03-01', 'evidence_items': 12}


## 2. Tres herramientas tipadas de solo lectura

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class SessionArgs(StrictModel):
    rival: str = Field(min_length=1)
    fecha_corte: date


class EvidenceArgs(StrictModel):
    evidence_ids: list[str] = Field(min_length=1, max_length=12)


class ToolSpec(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True, extra="forbid")
    name: str
    description: str
    args_model: type[BaseModel]
    handler: Callable[[BaseModel], dict[str, Any]]
    read_only: bool = True


class Budget(StrictModel):
    max_calls: int = Field(gt=0)
    max_seconds: float = Field(gt=0)
    max_session_tokens: int = Field(gt=0)


class AgentState(BaseModel):
    model_config = ConfigDict(extra="forbid", arbitrary_types_allowed=True)
    started_at: float
    calls_used: int = 0
    turns_used: int = 0
    input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0
    traces: list[dict[str, Any]] = Field(default_factory=list)
    errors: list[str] = Field(default_factory=list)
    failure_injection: set[str] = Field(default_factory=set)


class ToolError(RuntimeError): pass
class ScopeError(ToolError): pass
class BudgetError(ToolError): pass


def require_session(args):
    if args.rival != SESSION_RIVAL:
        raise ScopeError("rival_locked_to_session")
    if args.fecha_corte != SESSION_CUTOFF:
        raise ScopeError("cutoff_locked_to_session")


def obtener_historial(args: SessionArgs):
    require_session(args)
    return {
        "rival": SESSION_RIVAL,
        "fecha_corte": SESSION_CUTOFF.isoformat(),
        "history_match_ids": list(SESSION_MEMORY["history_match_ids"]),
        "n_partidos": len(SESSION_MEMORY["history_match_ids"]),
        "evidence_ids": ["L_SAMPLE"],
    }


def obtener_perfil_corners(args: SessionArgs):
    require_session(args)
    return {
        "rival": SESSION_RIVAL,
        "fecha_corte": SESSION_CUTOFF.isoformat(),
        "indicadores": list(SESSION_MEMORY["indicators"]),
        "limitaciones": list(SESSION_MEMORY["limitations"]),
        "modelos_promovidos": list(SESSION_MEMORY["promoted_models"]),
        "evidence_ids": sorted(SESSION_MEMORY["evidence_by_id"]),
    }


def consultar_evidencia(args: EvidenceArgs):
    missing = sorted(set(args.evidence_ids) - set(SESSION_MEMORY["evidence_by_id"]))
    if missing:
        raise ToolError("unknown_evidence_ids:" + ",".join(missing))
    return {
        "evidencia": [SESSION_MEMORY["evidence_by_id"][item] for item in args.evidence_ids],
        "evidence_ids": list(args.evidence_ids),
    }


TOOL_REGISTRY = {
    "obtener_historial": ToolSpec(
        name="obtener_historial",
        description="Devuelve los ocho partidos previos de la sesion bloqueada.",
        args_model=SessionArgs,
        handler=obtener_historial,
    ),
    "obtener_perfil_corners": ToolSpec(
        name="obtener_perfil_corners",
        description="Devuelve los indicadores ya calculados por 06 para la sesion bloqueada.",
        args_model=SessionArgs,
        handler=obtener_perfil_corners,
    ),
    "consultar_evidencia": ToolSpec(
        name="consultar_evidencia",
        description="Devuelve el detalle de evidence_ids existentes en la sesion.",
        args_model=EvidenceArgs,
        handler=consultar_evidencia,
    ),
}
assert set(TOOL_REGISTRY) == {
    "obtener_historial", "obtener_perfil_corners", "consultar_evidencia"
}
assert all(spec.read_only for spec in TOOL_REGISTRY.values())


def summarize_result(result):
    return {
        "keys": sorted(result),
        "row_counts": {
            key: len(value)
            for key, value in result.items()
            if isinstance(value, (list, tuple))
        },
    }


def invoke_tool(name, arguments, state, budget):
    started = time.perf_counter()
    trace = {
        "kind": "tool",
        "tool": name,
        "arguments": arguments,
        "arguments_validated": False,
        "status": "started",
    }
    try:
        if name not in TOOL_REGISTRY:
            raise ScopeError("tool_not_registered")
        spec = TOOL_REGISTRY[name]
        if not spec.read_only:
            raise ScopeError("tool_not_read_only")
        if state.calls_used >= budget.max_calls:
            raise BudgetError("tool_call_budget_exceeded")
        if time.perf_counter() - state.started_at >= budget.max_seconds:
            raise BudgetError("wall_time_budget_exceeded")
        if state.total_tokens >= budget.max_session_tokens:
            raise BudgetError("session_token_budget_exceeded")
        args = spec.args_model.model_validate(arguments)
        trace["arguments_validated"] = True
        state.calls_used += 1
        if name in state.failure_injection:
            raise ToolError("simulated_tool_failure")
        result = spec.handler(args)
        trace.update(status="ok", result_summary=summarize_result(result))
        return result
    except Exception as error:
        trace.update(status="error", error_type=type(error).__name__, error_code=str(error))
        state.errors.append(f"{type(error).__name__}:{error}")
        raise
    finally:
        trace["latency_ms"] = (time.perf_counter() - started) * 1000
        state.traces.append(trace)


display(pd.DataFrame([
    {
        "tool": spec.name,
        "args_schema": spec.args_model.__name__,
        "read_only": spec.read_only,
        "description": spec.description,
    }
    for spec in TOOL_REGISTRY.values()
]))

,tool,args_schema,read_only,description
0,obtener_historial,SessionArgs,True,Devuelve los ocho partidos previos de la sesio...
1,obtener_perfil_corners,SessionArgs,True,Devuelve los indicadores ya calculados por 06 ...
2,consultar_evidencia,EvidenceArgs,True,Devuelve el detalle de evidence_ids existentes...


## 3. Agente OpenAI acotado y fallback determinista

La ruta real permite como maximo dos turnos del proveedor. Cada function
call se valida con el modelo Pydantic de la herramienta y solo se ejecuta
mediante `invoke_tool`. Los resultados regresan al modelo como tool outputs
antes de solicitar `AgentAnswer`. Si OpenAI falla o agota un presupuesto,
se usa una respuesta determinista; nunca se encubre un mock como produccion.

In [ ]:
class AgentAnswer(StrictModel):
    status: Literal["answered", "out_of_scope", "error"]
    answer: str = Field(min_length=1, max_length=4000)
    evidence_ids: list[str] = Field(default_factory=list, max_length=12)
    tool_calls: int = Field(ge=0)


OUT_OF_SCOPE = [
    "sql", "insert", "update", "delete", "escribe", "archivo",
    "apuesta", "marcador", "navega", "internet", "web",
]
SYSTEM_INSTRUCTION = """Eres el agente historico de CornerScout para una sesion bloqueada.
Usa exclusivamente las tres herramientas declaradas. No uses conocimiento externo.
No ejecutes SQL, no accedas a archivos, no escribas, no navegues y no cambies rival o fecha.
Toda cifra debe aparecer literalmente en un resultado de herramienta. Toda respuesta
answered debe citar evidence_ids existentes devueltos por las herramientas. No inventes
cifras. Devuelve AgentAnswer y tool_calls debe ser el numero real de herramientas ejecutadas."""


def format_evidence(item):
    if "nombre" in item:
        return (
            f"{item['nombre']}: {item['numerador']}/{item['denominador']}, "
            f"valor={item['valor']}, referencia={item['referencia_liga_previa']}, "
            f"cobertura={item['cobertura']} [{item['evidence_id']}]"
        )
    return f"{item.get('texto', '')} [{item['evidence_id']}]"


def numeric_tokens(value):
    return set(re.findall(r"(?<![A-Za-z0-9_])\d+(?:[.,]\d+)?", json.dumps(value, ensure_ascii=False)))


def result_evidence_ids(tool_results):
    available = set()
    for result in tool_results:
        available.update(result.get("evidence_ids", []))
        for key in ("evidencia", "indicadores", "limitaciones", "modelos_promovidos"):
            for item in result.get(key, []):
                if isinstance(item, dict) and item.get("evidence_id"):
                    available.add(item["evidence_id"])
    return available


def validate_agent_answer(answer, state, tool_results, require_tool=False):
    answer = AgentAnswer.model_validate(answer)
    if answer.tool_calls != state.calls_used:
        raise ValueError("tool_calls_mismatch")
    if len(answer.evidence_ids) != len(set(answer.evidence_ids)):
        raise ValueError("duplicate_evidence_ids")
    known = set(SESSION_MEMORY["evidence_by_id"])
    if not set(answer.evidence_ids) <= known:
        raise ValueError("unknown_answer_evidence_id")
    if require_tool and state.calls_used == 0:
        raise ValueError("required_tool_not_called")
    if answer.status == "answered":
        if state.calls_used == 0 or not answer.evidence_ids:
            raise ValueError("answered_requires_tool_and_evidence")
        if not set(answer.evidence_ids) <= result_evidence_ids(tool_results):
            raise ValueError("evidence_not_returned_by_tool")
        allowed_numbers = set().union(*(numeric_tokens(item) for item in tool_results))
        unsupported = numeric_tokens(answer.answer) - allowed_numbers
        if unsupported:
            raise ValueError("unsupported_numbers:" + ",".join(sorted(unsupported)))
    return answer


def add_usage(state, response):
    usage = getattr(response, "usage", None)
    input_tokens = int(getattr(usage, "input_tokens", 0) or 0)
    output_tokens = int(getattr(usage, "output_tokens", 0) or 0)
    state.input_tokens += input_tokens
    state.output_tokens += output_tokens
    state.total_tokens += input_tokens + output_tokens
    state.traces.append({
        "kind": "provider",
        "turn": state.turns_used,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "response_id": getattr(response, "id", None),
    })


def ensure_budget(state, budget):
    if time.perf_counter() - state.started_at >= budget.max_seconds:
        raise BudgetError("wall_time_budget_exceeded")
    if state.total_tokens > budget.max_session_tokens:
        raise BudgetError("session_token_budget_exceeded")
    if state.calls_used > budget.max_calls:
        raise BudgetError("tool_call_budget_exceeded")
    if state.turns_used > MAX_REAL_TURNS:
        raise BudgetError("real_turn_budget_exceeded")


def deterministic_fallback(question, reason, budget=None, state=None):
    budget = budget or Budget(
        max_calls=MAX_TOOL_CALLS,
        max_seconds=MAX_WALL_SECONDS,
        max_session_tokens=MAX_SESSION_TOKENS,
    )
    state = state or AgentState(started_at=time.perf_counter())
    lowered = question.lower()
    if any(term in lowered for term in OUT_OF_SCOPE):
        answer = AgentAnswer(
            status="out_of_scope",
            answer="Solicitud fuera del alcance de la sesion de solo lectura.",
            evidence_ids=[],
            tool_calls=0,
        )
        return answer, state
    try:
        requested = sorted(set(re.findall(r"(?:E|L|M)_[A-Z0-9_]+", question.upper())))
        if requested:
            result = invoke_tool("consultar_evidencia", {"evidence_ids": requested}, state, budget)
            answer = AgentAnswer(
                status="answered",
                answer=" ".join(format_evidence(item) for item in result["evidencia"]),
                evidence_ids=requested,
                tool_calls=state.calls_used,
            )
            return validate_agent_answer(answer, state, [result], require_tool=True), state
        if any(term in lowered for term in ["partidos", "historial", "match_ids"]):
            result = invoke_tool(
                "obtener_historial",
                {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
                state,
                budget,
            )
            answer = AgentAnswer(
                status="answered",
                answer="history_match_ids=" + json.dumps(result["history_match_ids"]),
                evidence_ids=["L_SAMPLE"],
                tool_calls=state.calls_used,
            )
            return validate_agent_answer(answer, state, [result], require_tool=True), state
        result = invoke_tool(
            "obtener_perfil_corners",
            {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
            state,
            budget,
        )
        evidence_id = "E_HIGH" if "alto" in lowered else "E_SCR15" if "scr" in lowered else "E_CORNERS"
        item = next(value for value in result["indicadores"] if value["evidence_id"] == evidence_id)
        answer = AgentAnswer(
            status="answered",
            answer=format_evidence(item),
            evidence_ids=[evidence_id],
            tool_calls=state.calls_used,
        )
        return validate_agent_answer(answer, state, [result], require_tool=True), state
    except Exception as error:
        state.errors.append(f"fallback:{reason}:{type(error).__name__}:{error}")
        return AgentAnswer(
            status="error",
            answer="No fue posible responder con la evidencia disponible.",
            evidence_ids=[],
            tool_calls=state.calls_used,
        ), state


def run_mock_agent(question, budget=None, failure_injection=None):
    """Test-only deterministic harness; never used as the production fallback."""
    budget = budget or Budget(
        max_calls=MAX_TOOL_CALLS,
        max_seconds=MAX_WALL_SECONDS,
        max_session_tokens=MAX_SESSION_TOKENS,
    )
    state = AgentState(
        started_at=time.perf_counter(),
        failure_injection=failure_injection or set(),
    )
    answer, state = deterministic_fallback(question, "mock_test", budget, state)
    return answer, state


def openai_tools():
    return [{
        "type": "function",
        "name": spec.name,
        "description": spec.description,
        "parameters": spec.args_model.model_json_schema(),
        "strict": True,
    } for spec in TOOL_REGISTRY.values()]


def run_real_agent(question, require_tool=False, client=None):
    budget = Budget(
        max_calls=MAX_TOOL_CALLS,
        max_seconds=MAX_WALL_SECONDS,
        max_session_tokens=MAX_SESSION_TOKENS,
    )
    state = AgentState(started_at=time.perf_counter())
    tool_results = []
    tool_result_returned = False
    real_call = False
    try:
        if any(term in question.lower() for term in OUT_OF_SCOPE):
            raise ScopeError("question_out_of_scope")
        if not RUN_REAL_LLM or not OPENAI_API_KEY:
            raise RuntimeError("openai_not_configured")
        client = client or OpenAI(api_key=OPENAI_API_KEY, timeout=OPENAI_TIMEOUT_SECONDS)
        response = None
        previous_response_id = None
        pending_input = [
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": (
                f"SESSION rival={SESSION_RIVAL}; fecha_corte={SESSION_CUTOFF.isoformat()}. "
                f"Pregunta: {question}"
            )},
        ]
        while state.turns_used < MAX_REAL_TURNS:
            ensure_budget(state, budget)
            state.turns_used += 1
            real_call = True
            kwargs = {
                "model": MODEL_NAME,
                "input": pending_input,
                "tools": openai_tools(),
                "text_format": AgentAnswer,
                "max_output_tokens": MAX_OUTPUT_TOKENS,
            }
            if previous_response_id:
                kwargs["previous_response_id"] = previous_response_id
                tool_result_returned = bool(pending_input)
            remaining_seconds = budget.max_seconds - (time.perf_counter() - state.started_at)
            if remaining_seconds <= 0:
                raise BudgetError("wall_time_budget_exceeded")
            request_client = (
                client.with_options(timeout=min(OPENAI_TIMEOUT_SECONDS, remaining_seconds))
                if hasattr(client, "with_options") else client
            )
            response = request_client.responses.parse(**kwargs)
            add_usage(state, response)
            ensure_budget(state, budget)
            function_calls = [
                item for item in response.output
                if getattr(item, "type", None) == "function_call"
            ]
            if function_calls:
                outputs = []
                for call in function_calls:
                    arguments = json.loads(call.arguments)
                    result = invoke_tool(call.name, arguments, state, budget)
                    tool_results.append(result)
                    outputs.append({
                        "type": "function_call_output",
                        "call_id": call.call_id,
                        "output": json.dumps(result, ensure_ascii=False, default=str),
                    })
                previous_response_id = response.id
                pending_input = outputs
                continue
            parsed = response.output_parsed
            if parsed is None:
                parsed = AgentAnswer.model_validate_json(response.output_text)
            answer = validate_agent_answer(parsed, state, tool_results, require_tool=require_tool)
            return {
                "answer": answer,
                "state": state,
                "mode": "openai",
                "real_call": real_call,
                "tool_result_returned": tool_result_returned,
                "final_approved": True,
                "error": None,
            }
        raise BudgetError("real_turn_budget_exceeded")
    except Exception as error:
        state.errors.append(f"{type(error).__name__}:{error}")
        fallback, fallback_state = deterministic_fallback(question, str(error), budget, state)
        return {
            "answer": fallback,
            "state": state,
            "fallback_state": fallback_state,
            "mode": "fallback_deterministic",
            "real_call": real_call,
            "tool_result_returned": tool_result_returned,
            "final_approved": False,
            "error": f"{type(error).__name__}:{error}",
        }


default_answer, default_state = run_mock_agent("Muestra el historial de partidos usado.")
print(default_answer.model_dump())
display(pd.DataFrame(default_state.traces))

{'status': 'answered', 'answer': 'history_match_ids=[265839, 267273, 266815, 266254, 266160, 267576, 265894, 266149]', 'evidence_ids': ['L_SAMPLE'], 'tool_calls': 1}


,kind,tool,arguments,arguments_validated,status,result_summary,latency_ms
0,tool,obtener_historial,"{'rival': 'Barcelona', 'fecha_corte': '2016-03...",True,ok,"{'keys': ['evidence_ids', 'fecha_corte', 'hist...",0.101747


## 4. Pruebas mock operativas, de contrato y alcance

In [ ]:
test_rows = []

def record_test(name, expected_error, operation, predicate=None):
    try:
        result = operation()
        passed = expected_error is None and (predicate(result) if predicate else True)
        observed = "ok"
    except Exception as error:
        passed = expected_error is not None and isinstance(error, expected_error)
        observed = type(error).__name__ + ":" + str(error)
    test_rows.append({"test": name, "passed": passed, "observed": observed})

def fresh_state(**updates):
    return AgentState(started_at=time.perf_counter(), **updates)

default_budget = Budget(
    max_calls=MAX_TOOL_CALLS,
    max_seconds=MAX_WALL_SECONDS,
    max_session_tokens=MAX_SESSION_TOKENS,
)
record_test(
    "history_question", None,
    lambda: run_mock_agent("Muestra el historial de partidos usado."),
    lambda result: result[0].status == "answered" and result[0].tool_calls == 1,
)
record_test(
    "query_E_SHORT", None,
    lambda: run_mock_agent("Consulta E_SHORT"),
    lambda result: result[0].evidence_ids == ["E_SHORT"],
)
record_test(
    "compare_E_HIGH_reference", None,
    lambda: run_mock_agent("Consulta E_HIGH y su referencia"),
    lambda result: "referencia=" in result[0].answer and result[0].evidence_ids == ["E_HIGH"],
)
record_test(
    "query_L_MODEL", None,
    lambda: run_mock_agent("Consulta L_MODEL"),
    lambda result: result[0].evidence_ids == ["L_MODEL"],
)
record_test(
    "different_rival", ScopeError,
    lambda: invoke_tool(
        "obtener_historial",
        {"rival": "Real Madrid", "fecha_corte": SESSION_CUTOFF.isoformat()},
        fresh_state(), default_budget,
    ),
)
record_test(
    "different_date", ScopeError,
    lambda: invoke_tool(
        "obtener_perfil_corners",
        {"rival": SESSION_RIVAL, "fecha_corte": "2016-04-01"},
        fresh_state(), default_budget,
    ),
)
record_test(
    "unknown_evidence", ToolError,
    lambda: invoke_tool(
        "consultar_evidencia", {"evidence_ids": ["E_DOES_NOT_EXIST"]},
        fresh_state(), default_budget,
    ),
)
for name, question in [
    ("sql_request", "Ejecuta SQL SELECT * FROM corners"),
    ("write_request", "Escribe un archivo con el reporte"),
    ("betting_request", "Dame una apuesta y marcador"),
]:
    record_test(
        name, None, lambda question=question: run_mock_agent(question),
        lambda result: result[0].status == "out_of_scope" and result[0].tool_calls == 0,
    )

def exceed_calls():
    state = fresh_state()
    budget = Budget(max_calls=1, max_seconds=MAX_WALL_SECONDS, max_session_tokens=MAX_SESSION_TOKENS)
    arguments = {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()}
    invoke_tool("obtener_historial", arguments, state, budget)
    return invoke_tool("obtener_perfil_corners", arguments, state, budget)

record_test("excess_tool_calls", BudgetError, exceed_calls)
record_test(
    "simulated_timeout", BudgetError,
    lambda: invoke_tool(
        "obtener_historial",
        {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
        AgentState(started_at=time.perf_counter() - MAX_WALL_SECONDS - 1),
        default_budget,
    ),
)
record_test(
    "tool_failure", ToolError,
    lambda: invoke_tool(
        "obtener_perfil_corners",
        {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
        fresh_state(failure_injection={"obtener_perfil_corners"}),
        default_budget,
    ),
)

def invalid_final_answer():
    state = fresh_state(calls_used=1)
    result = {"evidence_ids": ["E_SHORT"], "evidencia": [SESSION_MEMORY["evidence_by_id"]["E_SHORT"]]}
    answer = AgentAnswer(
        status="answered", answer="Valor inventado 999.",
        evidence_ids=["E_SHORT"], tool_calls=1,
    )
    return validate_agent_answer(answer, state, [result], require_tool=True)

record_test("invalid_final_answer", ValueError, invalid_final_answer)

tests = pd.DataFrame(test_rows)
assert len(tests) == 14 and tests.passed.all()
display(tests)

real_questions = [
    {"question_id": "real_history", "question": "Muestra los partidos del historial de esta sesion."},
    {"question_id": "real_evidence", "question": "Consulta la evidencia E_SHORT de esta sesion."},
    {"question_id": "real_multi_tool", "question": "Compara el historial de partidos con el indicador E_HIGH y su referencia."},
    {"question_id": "real_out_of_scope", "question": "Ejecuta SQL SELECT * FROM corners"},
]
real_rows = []
real_run_results = []
for item in real_questions:
    started = time.perf_counter()
    result = run_real_agent(item["question"], require_tool=True)
    state = result["state"]
    arguments_validated = bool(state.calls_used) and all(
        trace.get("arguments_validated")
        for trace in state.traces
        if trace.get("kind") == "tool"
    )
    real_rows.append({
        "question_id": item["question_id"],
        "execution_mode": result["mode"],
        "real_call": result["real_call"],
        "input_tokens": state.input_tokens,
        "output_tokens": state.output_tokens,
        "total_tokens": state.total_tokens,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "turns": state.turns_used,
        "tool_calls": state.calls_used,
        "arguments_validated": arguments_validated,
        "tool_result_returned": result["tool_result_returned"],
        "final_approved": result["final_approved"],
        "error": result["error"],
    })
    real_run_results.append(result)
real_evaluation = pd.DataFrame(real_rows)
in_scope = real_evaluation[real_evaluation.question_id != "real_out_of_scope"]
out_of_scope = real_evaluation[real_evaluation.question_id == "real_out_of_scope"]
openai_validated = bool(
    len(in_scope) == 3
    and in_scope.real_call.all()
    and in_scope.tool_calls.ge(1).all()
    and in_scope.arguments_validated.all()
    and in_scope.tool_result_returned.all()
    and in_scope.final_approved.all()
    and in_scope.execution_mode.eq("openai").all()
    and in_scope.input_tokens.gt(0).all()
    and in_scope.output_tokens.gt(0).all()
    and in_scope.total_tokens.eq(in_scope.input_tokens + in_scope.output_tokens).all()
    and len(out_of_scope) == 1
    and (~out_of_scope.real_call).all()
    and out_of_scope.execution_mode.eq("fallback_deterministic").all()
    and out_of_scope.tool_calls.eq(0).all()
    and out_of_scope.error.str.contains("ScopeError").all()
)
display(real_evaluation)
print({"openai_validated": openai_validated, "real_tests": len(real_evaluation)})

,test,passed,observed
0,history_question,True,ok
1,query_E_SHORT,True,ok
2,compare_E_HIGH_reference,True,ok
3,query_L_MODEL,True,ok
4,different_rival,True,ScopeError:rival_locked_to_session
5,different_date,True,ScopeError:cutoff_locked_to_session
6,unknown_evidence,True,ToolError:unknown_evidence_ids:E_DOES_NOT_EXIST
7,sql_request,True,ok
8,write_request,True,ok
9,betting_request,True,ok


,question_id,execution_mode,real_call,input_tokens,output_tokens,total_tokens,latency_ms,turns,tool_calls,arguments_validated,tool_result_returned,final_approved,error
0,real_history,openai,True,989,122,1111,3181.713505,2,1,True,True,True,None
1,real_evidence,openai,True,1006,127,1133,3030.932070,2,1,True,True,True,None
2,real_multi_tool,openai,True,1910,307,2217,5228.573450,2,2,True,True,True,None
3,real_out_of_scope,fallback_deterministic,False,0,0,0,0.061986,0,0,False,False,False,ScopeError:question_out_of_scope


{'openai_validated': True, 'real_tests': 4}


## 5. Comparacion mock con reporte de un boton

In [ ]:
one_button_json = json.dumps(SESSION_MEMORY["one_button_report"], ensure_ascii=False)
comparison_questions = [
    {"question_id": "history_ids", "question": "Que partidos forman el historial?",
     "required": [str(item) for item in SESSION_MEMORY["history_match_ids"]]},
    {"question_id": "short_detail", "question": "Consulta E_SHORT", "required": ["E_SHORT"]},
    {"question_id": "high_reference", "question": "Cual es el perfil de pase alto y su referencia?",
     "required": ["E_HIGH", "referencia="]},
    {"question_id": "model_limit", "question": "Consulta L_MODEL", "required": ["L_MODEL"]},
]
comparison_rows = []
comparison_traces = []
comparison_tokens = 0
for item in comparison_questions:
    answer, state = run_mock_agent(item["question"])
    agent_pass = all(required in answer.answer for required in item["required"])
    button_pass = all(required in one_button_json for required in item["required"])
    comparison_rows.append({"question_id": item["question_id"], "agent_pass": agent_pass,
                            "one_button_pass": button_pass, "tool_calls": state.calls_used,
                            "agent_status": answer.status})
    comparison_traces.extend([{"question_id": item["question_id"], **trace} for trace in state.traces])
    comparison_tokens += state.total_tokens
comparison = pd.DataFrame(comparison_rows)
agent_score = float(comparison.agent_pass.mean())
button_score = float(comparison.one_button_pass.mean())
adds_value = agent_score > button_score
assessment = {
    "adds_value": adds_value, "agent_answer_rate": agent_score,
    "one_button_answer_rate": button_score,
    "conclusion": ("Aporta valor en preguntas de seguimiento trazables sobre historial y evidencia; "
                   "no reemplaza el reporte de un boton y agrega costo operacional de llamadas y trazas."
                   if adds_value else
                   "No demuestra valor adicional suficiente frente al reporte de un boton."),
}
display(comparison)
print(json.dumps(assessment, indent=2, ensure_ascii=False))

,question_id,agent_pass,one_button_pass,tool_calls,agent_status
0,history_ids,True,False,1,answered
1,short_detail,True,True,1,answered
2,high_reference,True,False,1,answered
3,model_limit,True,True,1,answered


{
  "adds_value": true,
  "agent_answer_rate": 1.0,
  "one_button_answer_rate": 0.5,
  "conclusion": "Aporta valor en preguntas de seguimiento trazables sobre historial y evidencia; no reemplaza el reporte de un boton y agrega costo operacional de llamadas y trazas."
}


## 6. Artefactos, trazas y contrato

In [ ]:
tool_registry_table = pd.DataFrame([
    {
        "tool": spec.name,
        "description": spec.description,
        "args_model": spec.args_model.__name__,
        "read_only": spec.read_only,
    }
    for spec in TOOL_REGISTRY.values()
])
mock_traces = pd.DataFrame([*default_state.traces, *comparison_traces])
real_traces = pd.DataFrame([
    {"question_id": row["question_id"], **trace}
    for row, result in zip(real_rows, real_run_results)
    for trace in result["state"].traces
])
operational_traces = pd.concat([mock_traces.assign(execution_mode="mock"),
                                real_traces.assign(execution_mode="openai_attempt")],
                               ignore_index=True, sort=False)
cost_log = real_evaluation.copy()

tables = {
    "tool_registry.parquet": tool_registry_table,
    "operational_traces.parquet": operational_traces,
    "security_tests.parquet": tests,
    "comparison.parquet": comparison,
    "cost_log.parquet": cost_log,
}
artifacts = []
for name, frame in tables.items():
    path = OUT / name
    frame.to_parquet(path, index=False)
    artifacts.append({
        "file": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "sha256": digest(path),
    })
write_json(OUT / "value_assessment.json", assessment)
artifacts.append({
    "file": "value_assessment.json",
    "sha256": digest(OUT / "value_assessment.json"),
})

observed_modes = sorted(set(real_evaluation.execution_mode) | {"mock"})
if real_evaluation.execution_mode.eq("fallback_deterministic").any():
    observed_modes.append("fallback_deterministic")
    observed_modes = sorted(set(observed_modes))
contract07 = {
    "stage": "07_agent",
    "contract_version": STAGE_VERSION,
    "run_id": RUN_ID,
    "source_06_contract": contract06["contract_version"],
    "source_06_run_id": contract06["run_id"],
    "session": {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
    "tools": list(TOOL_REGISTRY),
    "tool_count": len(TOOL_REGISTRY),
    "read_only": True,
    "budgets": {
        "max_tool_calls": MAX_TOOL_CALLS,
        "openai_timeout_seconds": OPENAI_TIMEOUT_SECONDS,
        "max_wall_seconds": MAX_WALL_SECONDS,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "max_session_tokens": MAX_SESSION_TOKENS,
        "max_real_turns": MAX_REAL_TURNS,
    },
    "run_real_llm": RUN_REAL_LLM,
    "openai_validated": openai_validated,
    "real_tests_required": 2,
    "real_tests_executed": int(real_evaluation.real_call.sum()),
    "execution_modes": observed_modes,
    "execution_mode_contract": ["mock", "openai", "fallback_deterministic"],
    "multi_agent": False,
    "vector_database": False,
    "agent_file_access": False,
    "security_tests_passed": int(tests.passed.sum()),
    "security_tests_total": len(tests),
    "value_assessment": assessment,
    "artifacts": artifacts,
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "pydantic": pydantic.__version__,
        "jsonschema": version("jsonschema"),
    },
}
write_json(OUT / "contract.json", contract07)
print(json.dumps({
    key: contract07[key]
    for key in [
        "tools", "security_tests_passed", "security_tests_total",
        "run_real_llm", "openai_validated", "execution_modes",
    ]
}, indent=2, ensure_ascii=False))

{
  "tools": [
    "obtener_historial",
    "obtener_perfil_corners",
    "consultar_evidencia"
  ],
  "security_tests_passed": 14,
  "security_tests_total": 14,
  "run_real_llm": true,
  "openai_validated": false,
  "execution_modes": [
    "fallback_deterministic",
    "mock",
    "openai"
  ]
}


## Conclusion

Las tres herramientas de solo lectura y las pruebas mock quedan separadas de
la ruta productiva. El agente real solo puede declararse validado cuando las
dos preguntas diseñadas completan function call, validacion Pydantic,
`invoke_tool`, retorno del resultado al modelo y `AgentAnswer` aprobado con
tokens reales registrados. Un texto directo sin herramienta no cuenta como
prueba aprobada. Ante cualquier fallo se usa un fallback determinista.

## Cambio 1 — Error `BadRequestError` por `temperature`

**Síntoma:** las dos primeras corridas reales fallaban con `Error code: 400 - "Unsupported parameter: 'temperature' is not supported with this model."`, cayendo siempre a `fallback_deterministic` antes de llegar a `execution_mode="openai"`.

**Causa:** `MODEL_NAME` es un modelo que no acepta `temperature` (típico de la familia de modelos de razonamiento, que fijan su propio muestreo).

**Cambio en `run_real_agent`:**

```python
kwargs = {
    "model": MODEL_NAME,
    "input": pending_input,
    "tools": openai_tools(),
    "text_format": AgentAnswer,
    "temperature": 0,      # ← eliminado
    "max_output_tokens": MAX_OUTPUT_TOKENS,
}
```

Se quitó la línea `"temperature": 0,`. No se tocó nada más del flujo (ni `openai_tools()`, ni el manejo de `function_call`, ni `validate_agent_answer`).

## Cambio 2 — Ampliación de `real_questions`

**Motivo:** las dos preguntas originales (`real_history`, `real_evidence`) solo ejercitaban una llamada a herramienta cada una. Faltaba evidencia real de dos rutas del código que hasta entonces solo se habían probado con el arnés mock.

**Se añadieron dos preguntas:**

```python
{"question_id": "real_multi_tool", "question": "Compara el historial de partidos con el indicador E_HIGH y su referencia."},
{"question_id": "real_out_of_scope", "question": "Ejecuta SQL SELECT * FROM corners"},
```

- `real_multi_tool`: obliga a encadenar dos llamadas a herramienta en el mismo turno de conversación, probando el ciclo `while`, `previous_response_id` y `function_call_output` contra el modelo real, no solo el mock.
- `real_out_of_scope`: confirma que el corto-circuito por `OUT_OF_SCOPE` ocurre antes de crear el cliente y sin gastar tokens (`real_call=False`).

## Cambio 3 — Fórmula `openai_validated`

**Síntoma:** con las cuatro preguntas ya corriendo correctamente, `openai_validated` seguía saliendo `False` pese a que las tres reales en alcance funcionaban bien y la de fuera de alcance se comportaba exactamente como debía.

**Causa:** la fórmula original exigía que **todas** las filas cumplieran `real_call=True` y `execution_mode="openai"`. Eso era correcto cuando solo había preguntas en alcance, pero dejó de serlo al agregar `real_out_of_scope`, que por diseño debe evitar la API.

**Cambio:** se separó la validación en dos grupos.

```python
in_scope = real_evaluation[real_evaluation.question_id != "real_out_of_scope"]
out_of_scope = real_evaluation[real_evaluation.question_id == "real_out_of_scope"]

openai_validated = bool(
    len(in_scope) == 3
    and in_scope.real_call.all()
    and in_scope.tool_calls.ge(1).all()
    and in_scope.arguments_validated.all()
    and in_scope.tool_result_returned.all()
    and in_scope.final_approved.all()
    and in_scope.execution_mode.eq("openai").all()
    and in_scope.input_tokens.gt(0).all()
    and in_scope.output_tokens.gt(0).all()
    and in_scope.total_tokens.eq(in_scope.input_tokens + in_scope.output_tokens).all()
    and len(out_of_scope) == 1
    and (~out_of_scope.real_call).all()
    and out_of_scope.execution_mode.eq("fallback_deterministic").all()
    and out_of_scope.tool_calls.eq(0).all()
    and out_of_scope.error.str.contains("ScopeError").all()
)
```

- **Grupo en alcance (3 preguntas):** exige llamada real, herramienta ejecutada, argumentos validados, resultado devuelto, respuesta aprobada, modo `"openai"` y conteo de tokens consistente.
- **Grupo fuera de alcance (1 pregunta):** exige exactamente lo contrario — sin llamada real, modo `fallback_deterministic`, cero herramientas y el error capturado como `ScopeError`.

## Resultado final

- **14/14 pruebas mock** siguen pasando sin cambios (scope bloqueado, evidencia inexistente, presupuestos, fallo de herramienta, respuesta inválida).
- **4/4 corridas reales** correctas: dos herramienta-simple, una multi-herramienta y una fuera de alcance.
- `openai_validated=True` con la fórmula corregida.

El notebook 07 queda con evidencia real y mock suficiente para declararlo validado.